# D7: external GPU encoder training

Use a free GPU runtime with Python 3.11. This notebook has not been run. Upload only an exported metadata index and reduced logs for verified training/validation variants. Never upload test logs. GPU training is intentionally not performed on the development laptop.


In [ ]:
import sys

assert sys.version_info[:2] == (3, 11), "Select a Python 3.11 runtime"
!git clone --branch d7-encoder --single-branch https://github.com/trad024/FlowGuard.git
%cd FlowGuard
!python -m pip install -e .
!python -m pip install -r ci/encoder/requirements.txt

Prepare `/content/train-val/index.jsonl` and `/content/train-val/<variant>/reduced.log`. Each index row has only `variant_id` and `log_path`. The training script checks the committed split checksum and rejects test, unknown, duplicate or unverified variants before opening any log. No training data is downloaded automatically.


In [ ]:
import subprocess

subprocess.run(
    [
        "python",
        "ci/train_encoder.py",
        "--index",
        "/content/train-val/index.jsonl",
        "--log-root",
        "/content/train-val",
        "--output",
        "/content/encoder-output",
    ],
    check=True,
)

After a successful run, retain `validation-metrics.json`, `validation-predictions.json`, `checkpoint/encoder_metadata.json` and `artifact.json`. Store `encoder-weights.zip` outside git. Publish that ZIP to an approved artifact location and use its actual SHA-256 from `artifact.json` with `ci/download_encoder.py`. Run `ci/benchmark_encoder.py` on the target CPU laptop; report the measured RAM and latency before adopting the encoder. No result is established until those runs complete.
